## Following kernels will load source name text files into a filtered ARCHS4 database

### Function to load sample names from text file

In [1]:
import glob
import sys
import h5py
import archs4py as a4
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
def load_sample_names(txt_file_path):
    """Load sample names from a text file"""
    with open(txt_file_path, 'r') as f:
        sample_names = [line.strip() for line in f if line.strip()]
    return sample_names

In [4]:
%cd /scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/
class_sample_dict = dict()

# loop through all file paths in current directory to find files of interest
for filename in glob.glob("*3.txt"):
    file_path = "/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/" + filename
    # sample_names are within the text files and are the values of the class_sample_dict
    sample_names = load_sample_names(file_path)
    # class names are within the names of the text files
    class_name = filename.replace("3.txt","")

    class_sample_dict[class_name] = sample_names

/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class


In [5]:
class_sample_dict

{'Macrophage': ['PCA-ASSOCIATED CD14+CD11B+ MACROPHAGES',
  'WT ESC-DERIVED MACROPHAGE',
  'PERIPHERAL BLOOD MONOCYTE-DERIVED MACROPHAGES',
  'PRIMARY MACROPHAGE',
  'PRIMARY HUMAN MONOCYTE-DERIVED MACROPHAGES, ADMAF, IFN-Î³',
  'MACROPHAGE WITH CONTROL LNA 24H_REP#2',
  'MACROPHAGE_6D_MEDIA_MTB_4H',
  'PRIMARY HUMAN MACROPHAGE CELLS',
  'H1 ESC-DERIVED M1 MACROPHAGES',
  'MACROPHAGES(BC9)',
  'HUMAN MONOCYTE-DERIVED MACROPHAGE',
  'ACTIVATED MONOCYTE-DERIVED MACROPHAGES',
  'PERIPHERAL_BLOOD_MACROPHAGE-M0_SICTRL',
  'IPSC-DERIVED M0 MACROPHAGE',
  'HU-CB-MACROPHAGE-RNA, WT-CB',
  'BONE MARROW DERIVED MACROPHAGE',
  'HUMAN HEALTHY BLOOD DONOR MONOCYTES-DERIVED MACROPHAGES',
  'MACROPHAGES',
  'MONOCYTE-DERIVED MACROPHAGES, SSC',
  'PERIPHERAL BLOOD MONOCYTE-DERIVED MACROPHAGES NATURAL',
  'MACROPHAGE IN NORMOXIA 48H',
  'RNA FROM MONOCYTE-DERIVED MACROPHAGES 4 HOURS AFTER CULTURED WITHOUT VIRUS',
  'HUMAN IPSCS-DERIVED MACROPHAGES',
  'PRIMARY HUMAN MONOCYTE-DERIVED MACROPHAGES',
  'HU

### Filters human_tpm_v2.2_NOSC.h5 into a filtered file containing source names in class_sample_dict

In [6]:
transcript_file_path = "/scratch/by2372/pipeline/ARCHS4_Macrophages/human_tpm_v2.2_NOSC.h5"
a4.ls(transcript_file_path)

data                      
│ expression            float32 | (271742, 661840)
meta                      
│ samples                   
│   characteristics_ch1   str    | (661840,)
│   source_name_ch1       str    | (661840,)
│ transcripts               
│   ensembl_id            str    | (271742,)
│   symbol                str    | (271742,)


In [7]:

with h5py.File(transcript_file_path, "r") as fin, h5py.File('filtered_ARCHS4_4_class.h5', 'w') as fout:

    # copy the ensembl_ids and gene symbols to the new h5 file
    fin.copy("meta/transcripts/ensembl_id", fout)
    fin.copy("meta/transcripts/symbol", fout) 
    
    # Navigate to the metadata group for source names
    source_name_ds = fin['meta/samples/source_name_ch1']

    # Read all names and decode bytes to strings
    source_names = source_name_ds[:]
    if isinstance(source_names[0], bytes):
        source_names = [s.decode('utf-8') for s in source_names]

    # Get dataset info
    expr_dataset = fin['data/expression']
    n_genes, n_samples = expr_dataset.shape

    filtered_dataset = fout.create_dataset(
        "data/filtered_expression",
        (n_genes, 0),                    # start with 0 samples
        maxshape=(n_genes, None),        # allow dynamic extension
        dtype=expr_dataset.dtype,
        chunks=True
    )
    
    class_name_dataset = fout.create_dataset(
        "class_name",
        (0,),                            # start empty
        maxshape=(None,),                # allow appending
        dtype=source_name_ds.dtype,
        chunks=True
    )

    current_pos = 0

    for key, value in class_sample_dict.items():

        print("Matching indices for " + str(key))

        
        matched_indices = []
        class_names = []
        
        for filt_name in set(value): 
            matched_indices.extend([
                i for i, n in enumerate(source_names)
                if n.lower() == filt_name.lower()
            ])

        matched_indices.sort()
        n_new = len(matched_indices)
        print(str(n_new))
        cur_class_names = [key] * n_new
        class_names.append(cur_class_names)

        filtered_dataset.resize((n_genes, current_pos + n_new))
        class_name_dataset.resize((current_pos + n_new,))
        
        # Process in chunks of genes instead of loading all at once
        chunk_size = 1000  # Adjust based on your memory
        iteration = 0
        for start_gene in range(0, n_genes, chunk_size):
            end_gene = min(start_gene + chunk_size, n_genes)
            
            # Read chunk of data for matched samples only
            chunk_data = expr_dataset[start_gene:end_gene, matched_indices]
            
            # Write to output
            filtered_dataset[start_gene:end_gene, current_pos:current_pos + n_new] = chunk_data
            iteration = iteration + 1
            print("Iteration " + str(iteration)+ " is done.")
            sys.stdout.flush()


        class_name_dataset[current_pos:current_pos + n_new] = class_names
        
        current_pos += n_new

Matching indices for Macrophage
4183
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Iteration 10 is done.
Iteration 11 is done.
Iteration 12 is done.
Iteration 13 is done.
Iteration 14 is done.
Iteration 15 is done.
Iteration 16 is done.
Iteration 17 is done.
Iteration 18 is done.
Iteration 19 is done.
Iteration 20 is done.
Iteration 21 is done.
Iteration 22 is done.
Iteration 23 is done.
Iteration 24 is done.
Iteration 25 is done.
Iteration 26 is done.
Iteration 27 is done.
Iteration 28 is done.
Iteration 29 is done.
Iteration 30 is done.
Iteration 31 is done.
Iteration 32 is done.
Iteration 33 is done.
Iteration 34 is done.
Iteration 35 is done.
Iteration 36 is done.
Iteration 37 is done.
Iteration 38 is done.
Iteration 39 is done.
Iteration 40 is done.
Iteration 41 is done.
Iteration 42 is done.
Iteration 43 is done.
Iteration 44 is done.
Iter

# Following code will filter the database for TF genes only

In [8]:
filtered_file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_ARCHS4_4_class.h5"
tf_names = load_sample_names("/scratch/by2372/pipeline/tf_names.txt")
a4.ls(filtered_file)

class_name            str    | (10775,)
data                      
│ filtered_expression   float32 | (271742, 10775)
ensembl_id            str    | (271742,)
symbol                str    | (271742,)


In [9]:
# Read in the filtered ARCHS4 h5 and write out to a new h5
with h5py.File(filtered_file, "r") as fin, h5py.File('filtered_ARCHS4_TFs_4_class.h5', 'w') as fout:

    # Copy the class names into the new h5
    fin.copy("class_name", fout) 
    # Read in the gene symbol and ensembl IDs from the input h5 file
    symbols = fin["symbol"]
    ensembl_id = fin["ensembl_id"]

    # unpack the gene symbol names and decode bytes to strings
    symbols_unpack = symbols[:]
    if isinstance(symbols_unpack[0], bytes):
        symbols_unpack = [s.decode('utf-8') for s in symbols_unpack]

    # unapck the ensembl IDs and decode bytes to strings
    ensembl_unpack = ensembl_id[:]
    if isinstance(ensembl_unpack[0], bytes):
        ensembl_unpack = [s.decode('utf-8') for s in ensembl_unpack]

    # initialize the total number of genes in the input h5, an empty matched index list and the processing chunk size
    n_genes = len(symbols)
    tf_matched_indices = []
    chunk_size = 1000

    # iterate through indices of all genes in the input h5 by chunking through them
    for start_gene in range(0, n_genes, chunk_size):
        end_gene = min(start_gene + chunk_size, n_genes)
        chunk_symbols = symbols_unpack[start_gene:end_gene]

        # find the indices that match the list of known human TFs
        for i, sym in enumerate(chunk_symbols):
            if sym in tf_names:
                idx = start_gene + i
                tf_matched_indices.append(idx)

    tf_matched_indices.sort()
    print("Matched indices for TFs have been found")
    
    # Create output dataset for gene symbol in the output h5
    new_sym_dataset = fout.create_dataset(
        "symbol", 
        (len(tf_matched_indices), ),
        dtype=symbols.dtype,
        chunks=True
    )

    # Create output dataset for ensembl ID in the output h5
    new_ensembl_dataset = fout.create_dataset(
        "ensembl_id", 
        (len(tf_matched_indices), ),
        dtype=ensembl_id.dtype,
        chunks=True
    )

    # convert symbols and ensembl IDs to np arrays
    symbols_arr = np.array(symbols_unpack)
    ensembl_arr = np.array(ensembl_unpack)

    # have only the matched indices be the new symbols and ensembl ID datasets
    new_sym_dataset[...] = symbols_arr[tf_matched_indices]
    new_ensembl_dataset[...] = ensembl_arr[tf_matched_indices]

    # Get dataset info
    expr_dataset = fin['data/filtered_expression']
    n_genes, n_samples = expr_dataset.shape

    # Create output dataset
    filtered_dataset = fout.create_dataset(
        "data/filtered_expression", 
        (len(tf_matched_indices), n_samples),
        dtype=expr_dataset.dtype,
        chunks=True
    )

    # do a chunking approach to chunk through the number of samples in 1000 chunks 
    iteration = 0
    for start_sample in range(0, n_samples, chunk_size):
        end_sample = min(start_sample + chunk_size, n_samples)
        data_chunk = expr_dataset[tf_matched_indices, start_sample:end_sample]
        filtered_dataset[:, start_sample:end_sample] = data_chunk
        print("Iteration " + str(iteration)+ " is done.")
        iteration += 1


Matched indices for TFs have been found
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
Iteration 9 is done.
Iteration 10 is done.


In [10]:
filtered_TFs_file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_ARCHS4_TFs_4_class.h5"
a4.ls(filtered_TFs_file)

class_name            str    | (10775,)
data                      
│ filtered_expression   float32 | (13146, 10775)
ensembl_id            str    | (13146,)
symbol                str    | (13146,)


# The code below will create training and testing datasets

In [11]:
def chunk_samples(chunk_size, n_samples, n_genes, output, input):
    iteration = 0
    for start_sample in range(0, n_samples, chunk_size):
        end_sample = min(start_sample + chunk_size, n_samples)

        for start_gene in range(0, n_genes, chunk_size):
            end_gene = min(start_gene + chunk_size, n_genes)

            # Read a chunk of the input matrix
            data_chunk = input[start_sample:end_sample, start_gene:end_gene]

            # Write the chunk to the output dataset
            output[start_sample:end_sample, start_gene:end_gene] = data_chunk

        print("Iteration " + str(iteration) + " (samples " + str(start_sample) + "-" + str(end_sample) + ") is done.")
        iteration += 1

In [12]:
def chunk_class(chunk_size,n_samples,output,input):
    iteration = 0
    for start_sample in range(0, n_samples, chunk_size):
        end_sample = min(start_sample + chunk_size, n_samples)
        data_chunk = input[start_sample:end_sample]
        output[start_sample:end_sample] = data_chunk
        print("Iteration " + str(iteration)+ " is done.")
        iteration += 1

In [13]:
# Make Rows be the samples and the columns be the transcripts
# make it into a data frame and export as H5
with h5py.File(filtered_TFs_file, "r") as fin, h5py.File('filtered_training_4_class.h5', 'w') as ftrain, h5py.File('filtered_testing_4_class.h5', 'w') as ftest:

    # Navigate to the metadata group for class names, gene symbols, and ensembl ID
    class_name_ds = fin['class_name']
    ensembl_id = fin["ensembl_id"]
    symbols = fin["symbol"]

    # Read all names and decode bytes to strings
    symbols_unpack = symbols[:]
    if isinstance(symbols_unpack[0], bytes):
        symbols_unpack = [s.decode('utf-8') for s in symbols_unpack]

    # Read all names and decode bytes to strings
    class_names = class_name_ds[:]
    if isinstance(class_names[0], bytes):
        class_names = [s.decode('utf-8') for s in class_names]

    ensembl_unpack = ensembl_id[:]
    if isinstance(ensembl_unpack[0], bytes):
        ensembl_unpack = [s.decode('utf-8') for s in ensembl_unpack]

    # Navigate to the metadata group for the expression data
    expression = fin["data/filtered_expression"]

    # Transpose expression data so that rows are samples and columns are transcripts
    expression_data = expression[:, :] 
    expression_data = expression_data.T
    print("Expression data has been transposed\n")

    # Make the transcript ID be the symbol _ ensembl ID
    transcript_ID = []
    for idx, sym in enumerate(symbols_unpack):
        ID = sym + "_" + ensembl_unpack[idx]
        transcript_ID.append(ID)

    filtered_df = pd.DataFrame(expression_data,index = class_names, columns = transcript_ID)
    filtered_df['cell_identity'] = class_names

    # Count occurrences of each cell_identity
    cell_identity_counts = filtered_df['cell_identity'].value_counts()
    print(cell_identity_counts)

    # Create a boolean mask to filter rows with cell_identity occurring more than once
    mask = filtered_df['cell_identity'].isin(cell_identity_counts[cell_identity_counts > 1].index)

    # Filter the DataFrame using the mask
    filtered_df = filtered_df[mask]

    # Seperate the ensmbl ids from the cell_identity column
    TFs = filtered_df.columns.drop(['cell_identity'])

    # Set predictors and predicted
    X = np.array(filtered_df[TFs])
    y = np.array(filtered_df['cell_identity'])

    # Train - test split with stratification and shuffling
    ### THIS LOADS IN THE WHOLE H5 FILE TO SPLIT
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=5, stratify=y,shuffle = True)

    # Create output dataset
    X_train_out = ftrain.create_dataset(
        "expression", 
        (X_train.shape[0], X_train.shape[1]),
        dtype=expression.dtype,
        chunks=True
    )
    print("Starting X Train output into H5")
    chunk_samples(1000,X_train.shape[0],X_train.shape[1],X_train_out,X_train)
    print("X Train output successfully")

    y_train_out = ftrain.create_dataset(
        "label",
        len(y_train),
        dtype=h5py.string_dtype(encoding='utf-8'),
        chunks = True
    )
    print("Starting y train output into H5")
    chunk_class(1000,len(y_train),y_train_out,y_train)
    print("y Train output successfully")

    TF_ID_out = ftrain.create_dataset(
        "TF_ID",
        len(transcript_ID),
        dtype=h5py.string_dtype(encoding='utf-8'),
        chunks = True
    )
    print("Starting TF IDs output into training H5")
    chunk_class(1000,len(transcript_ID),TF_ID_out,transcript_ID)
    print("TF ID output successfully")
    
    ###export X_test, y_test with Transcript_ID or TFs to an h5
    
    X_test_out = ftest.create_dataset(
        "expression", 
        (X_test.shape[0], X_test.shape[1]),
        dtype=expression.dtype,
        chunks=True
    )
    print("Starting X Test output into H5")
    chunk_samples(1000,X_test.shape[0],X_test.shape[1],X_test_out,X_test)
    print("X test output successfully")

    y_test_out = ftest.create_dataset(
        "label",
        len(y_test),
        dtype=h5py.string_dtype(encoding='utf-8'),
        chunks = True
    )
    print("Starting y test output into H5")
    chunk_class(1000,len(y_test),y_test_out,y_test)
    print("y test output successfully")

    TF_ID_out = ftest.create_dataset(
        "TF_ID",
        len(transcript_ID),
        dtype=h5py.string_dtype(encoding='utf-8'),
        chunks = True
    )
    print("Starting TF IDs output into testing H5")
    chunk_class(1000,len(transcript_ID),TF_ID_out,transcript_ID)
    print("TF ID output successfully")



Expression data has been transposed

cell_identity
Monocyte      5497
Macrophage    4183
Microglia      672
other          423
Name: count, dtype: int64
Starting X Train output into H5
Iteration 0 (samples 0-1000) is done.
Iteration 1 (samples 1000-2000) is done.
Iteration 2 (samples 2000-3000) is done.
Iteration 3 (samples 3000-4000) is done.
Iteration 4 (samples 4000-5000) is done.
Iteration 5 (samples 5000-6000) is done.
Iteration 6 (samples 6000-7000) is done.
Iteration 7 (samples 7000-8000) is done.
Iteration 8 (samples 8000-8081) is done.
X Train output successfully
Starting y train output into H5
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iteration 6 is done.
Iteration 7 is done.
Iteration 8 is done.
y Train output successfully
Starting TF IDs output into training H5
Iteration 0 is done.
Iteration 1 is done.
Iteration 2 is done.
Iteration 3 is done.
Iteration 4 is done.
Iteration 5 is done.
Iterat

In [14]:
train_file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_training_4_class.h5"
#redownload the most recent transcript file
a4.ls(train_file)

TF_ID                 str    | (13146,)
expression            float32 | (8081, 13146)
label                 str    | (8081,)


In [15]:
test_file = "/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_testing_4_class.h5"
#redownload the most recent transcript file
a4.ls(test_file)

TF_ID                 str    | (13146,)
expression            float32 | (2694, 13146)
label                 str    | (2694,)


In [16]:
pd.Series(y_train).value_counts()

Monocyte      4123
Macrophage    3137
Microglia      504
other          317
Name: count, dtype: int64

In [17]:
pd.Series(y_test).value_counts()

Monocyte      1374
Macrophage    1046
Microglia      168
other          106
Name: count, dtype: int64

### Load in training and testing data here 

In [2]:
f_train = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_training_4_class.h5", 'r')
expression_train = f_train['expression'][...]
label_train = f_train['label'][...]
f_train.close()
f_test = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_testing_4_class.h5", 'r')
expression_test = f_test['expression'][...]
label_test = f_test['label'][...]

TFs = f_test['TF_ID'][...]

# Read all names and decode bytes to strings
if isinstance(label_train[0], bytes):
    label_train = [s.decode('utf-8') for s in label_train]

# Read all names and decode bytes to strings
if isinstance(label_test[0], bytes):
    label_test = [s.decode('utf-8') for s in label_test]

# Read all names and decode bytes to strings
if isinstance(TFs[0], bytes):
    TFs = [s.decode('utf-8') for s in TFs]

f_test.close()

### Random Oversampler on Training Data

In [3]:
from imblearn.over_sampling import RandomOverSampler
from collections import Counter

ros = RandomOverSampler(random_state=42)
expression_ros, label_ros = ros.fit_resample(expression_train,label_train)

print("Original class distribution:", Counter(label_train))
print("Resampled class distribution:", Counter(label_ros))

Original class distribution: Counter({'Monocyte': 4123, 'Macrophage': 3137, 'Microglia': 504, 'other': 317})
Resampled class distribution: Counter({'Monocyte': 4123, 'Macrophage': 4123, 'Microglia': 4123, 'other': 4123})


### SMOTE on Training Data

In [4]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=0,k_neighbors = 1)
expression_SMOTE, label_SMOTE = smote.fit_resample(expression_train,label_train)

print("Original class distribution:", Counter(label_train))
print("Resampled class distribution:", Counter(label_SMOTE))

Original class distribution: Counter({'Monocyte': 4123, 'Macrophage': 3137, 'Microglia': 504, 'other': 317})
Resampled class distribution: Counter({'Monocyte': 4123, 'Macrophage': 4123, 'Microglia': 4123, 'other': 4123})


### BLSMOTE on Training Data

In [5]:
from imblearn.over_sampling import BorderlineSMOTE
from collections import Counter

BLsmote = BorderlineSMOTE(random_state=1,k_neighbors = 1)
expression_BLSMOTE, label_BLSMOTE = BLsmote.fit_resample(expression_train,label_train)

print("Original class distribution:", Counter(label_train))
print("Resampled class distribution:", Counter(label_BLSMOTE))

Original class distribution: Counter({'Monocyte': 4123, 'Macrophage': 3137, 'Microglia': 504, 'other': 317})
Resampled class distribution: Counter({'Monocyte': 4123, 'Macrophage': 4123, 'Microglia': 4123, 'other': 4123})


# Below is the code for neural network training

In [2]:
from tensorflow import keras
from sklearn.preprocessing import LabelEncoder
import tensorflow as tef
from keras import optimizers
from keras import losses
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, KernelPCA
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (auc, roc_curve, roc_auc_score)
from tensorflow.keras.models import load_model
import os

2026-03-18 00:26:26.147884: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-18 00:26:29.770467: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-18 00:26:40.392075: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [7]:
def NN_training(X_train,y_train,X_test,y_test,TFS,name):

    keras.utils.set_random_seed(10)   
    
    num_unique_y = len(set(y_train))

    le = LabelEncoder()
    
    # Fit on training labels (strings) and transform them to integers
    label_train_int = le.fit_transform(y_train)
    
    # Use the same encoder to transform test labels
    label_test_int = le.transform(y_test)
    
    # Best model in magnussen paper was 2 layers of 250 nodes
    model = keras.Sequential([
        keras.layers.Input(shape=(len(TFs),)),  # there are 13146 TFs
        keras.layers.Dense(250, activation='elu'),
        keras.layers.Dense(250, activation='elu'),
        keras.layers.Dense(num_unique_y, activation='softmax')  # For multiclass classification
    ])

    callback = keras.callbacks.EarlyStopping(monitor='val_loss', patience = 3)
    
    adam = optimizers.Adam(
        learning_rate=0.001,
        beta_1=0.9,
        beta_2=0.999,
        weight_decay=0.01  # replaces old `decay`
    )
    model.compile(loss='sparse_categorical_crossentropy',optimizer=adam,metrics=['accuracy'], )
    model.fit(X_train, label_train_int, batch_size=50, epochs=100, verbose=1, validation_split = 0.1, callbacks = [callback])

    model.save("/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/" + name + "_trained_NN.keras")

    np.save("/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/" + name + "_class_names.npy", le.classes_)

    loss, accuracy = model.evaluate(X_test, label_test_int, verbose=1)

    # Predictions on test set
    y_prob = model.predict(X_test)
    y_pred = np.argmax(y_prob, axis=1)

    # Calculate F1 Metrics
    f1_macro = f1_score(label_test_int, y_pred, average='macro')
    f1_weighted = f1_score(label_test_int, y_pred, average='weighted')
    f1_micro = f1_score(label_test_int,y_pred,average = 'micro')

     # binarize the classes
    cell_classes = np.unique(np.concatenate((label_train_int, label_test_int)))
    y_test_bin = label_binarize(label_test_int, classes=cell_classes)

    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    roc_auc_labeled = dict()

    # One vs. rest AUC calculation for multiclass data
    for i in range(len(cell_classes)):
        cur_class = cell_classes[i]
        fpr[cur_class], tpr[cur_class], _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
        roc_auc[cur_class] = auc(fpr[cur_class], tpr[cur_class])
        # Inverse transform the label encoder to get the class label once again
        class_label = le.inverse_transform([cur_class])[0]
        roc_auc_labeled[class_label] = roc_auc[cur_class]
        print(f"AUC for class {cur_class}: {roc_auc[cur_class]:.2f}")

    # compute micro-average ROC curve and ROC area
    fpr["micro"], tpr["micro"],_=roc_curve(y_test_bin.ravel(),y_prob.ravel())
    roc_auc_labeled["micro"] = auc(fpr["micro"],tpr["micro"])   

    return {
    "f1_macro": f1_macro,
    "f1_weighted": f1_weighted,
    "f1_micro": f1_micro,
    "roc_auc_micro": roc_auc_labeled["micro"],
    "roc_auc_per_class": roc_auc_labeled,
    "loss" : loss,
    "accuracy" : accuracy
    }

In [8]:
unbal_metrics = NN_training(expression_train,label_train,expression_test,label_test,TFs,"unbalanced_4_class")


2026-03-17 00:19:35.832810: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.8769 - loss: 1.7023 - val_accuracy: 0.9234 - val_loss: 0.6159
Epoch 2/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9586 - loss: 0.1989 - val_accuracy: 0.9320 - val_loss: 0.5231
Epoch 3/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9629 - loss: 0.1881 - val_accuracy: 0.9654 - val_loss: 0.2159
Epoch 4/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9728 - loss: 0.1282 - val_accuracy: 0.9592 - val_loss: 0.2414
Epoch 5/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9824 - loss: 0.0815 - val_accuracy: 0.9543 - val_loss: 0.3616
Epoch 6/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9904 - loss: 0.0407 - val_accuracy: 0.9703 - val_loss: 0.1786
Epoch 7/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 2s 12ms/step - accuracy: 0.9897 - loss: 0.0482 - val_accuracy: 0.9728 - val_loss: 0.2654
Epoch 8/100
146/146 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9867 - loss: 0.0543 - 

In [9]:
unbal_metrics

{'f1_macro': 0.9622212889791782,
 'f1_weighted': 0.9743651759502449,
 'f1_micro': 0.9743875278396437,
 'roc_auc_micro': 0.9969425305981178,
 'roc_auc_per_class': {np.str_('Macrophage'): 0.99573821446472,
  np.str_('Microglia'): 0.999615899408061,
  np.str_('Monocyte'): 0.994777744695867,
  np.str_('other'): 0.9832827855705579,
  'micro': 0.9969425305981178},
 'loss': 0.25549188256263733,
 'accuracy': 0.974387526512146}

In [10]:
ROS_metrics = NN_training(expression_ros,label_ros,expression_test,label_test,TFs,"ROS_4_class")


Epoch 1/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9198 - loss: 0.7974 - val_accuracy: 0.9952 - val_loss: 0.0502
Epoch 2/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9691 - loss: 0.1837 - val_accuracy: 1.0000 - val_loss: 0.0067
Epoch 3/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9870 - loss: 0.0584 - val_accuracy: 1.0000 - val_loss: 0.0042
Epoch 4/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9922 - loss: 0.0351 - val_accuracy: 1.0000 - val_loss: 0.0058
Epoch 5/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9929 - loss: 0.0330 - val_accuracy: 0.9745 - val_loss: 0.0519
Epoch 6/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9755 - loss: 0.1680 - val_accuracy: 0.9933 - val_loss: 0.0554
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9733 - loss: 0.3570     
85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
AUC for class 0: 0.99
AUC for class 1: 1.00
AUC for class 2: 0.99
AUC for class 3: 0.98


In [11]:
ROS_metrics

{'f1_macro': 0.9695628857220243,
 'f1_weighted': 0.9732093006265568,
 'f1_micro': 0.9732739420935412,
 'roc_auc_micro': 0.9960527091741719,
 'roc_auc_per_class': {np.str_('Macrophage'): 0.9945922051643802,
  np.str_('Microglia'): 0.9999104550767257,
  np.str_('Monocyte'): 0.9928124035110935,
  np.str_('other'): 0.9790469802572104,
  'micro': 0.9960527091741719},
 'loss': 0.35704711079597473,
 'accuracy': 0.9732739329338074}

In [12]:
SMOTE_metrics = NN_training(expression_SMOTE,label_SMOTE,expression_test,label_test,TFs,"SMOTE_4_class")

Epoch 1/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9287 - loss: 0.7324 - val_accuracy: 1.0000 - val_loss: 0.0107
Epoch 2/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9746 - loss: 0.1761 - val_accuracy: 0.9442 - val_loss: 0.1681
Epoch 3/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9864 - loss: 0.0630 - val_accuracy: 1.0000 - val_loss: 0.0016
Epoch 4/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9795 - loss: 0.1265 - val_accuracy: 1.0000 - val_loss: 0.0038
Epoch 5/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9825 - loss: 0.1356 - val_accuracy: 1.0000 - val_loss: 0.0032
Epoch 6/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9912 - loss: 0.0466 - val_accuracy: 1.0000 - val_loss: 6.4603e-04
Epoch 7/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9969 - loss: 0.0112 - val_accuracy: 1.0000 - val_loss: 4.6807e-04
Epoch 8/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9966 - loss: 0

In [13]:
SMOTE_metrics

{'f1_macro': 0.9845032708690801,
 'f1_weighted': 0.9892172800169263,
 'f1_micro': 0.9892353377876764,
 'roc_auc_micro': 0.9969269837543059,
 'roc_auc_per_class': {np.str_('Macrophage'): 0.9951752167294734,
  np.str_('Microglia'): 1.0,
  np.str_('Monocyte'): 0.9956304309470247,
  np.str_('other'): 0.9862245195532354,
  'micro': 0.9969269837543059},
 'loss': 0.32955214381217957,
 'accuracy': 0.9892353415489197}

In [15]:
BLSMOTE_metrics = NN_training(expression_BLSMOTE,label_BLSMOTE,expression_test,label_test,TFs,"BLSMOTE_4_class")

Epoch 1/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9280 - loss: 0.7257 - val_accuracy: 1.0000 - val_loss: 0.0205
Epoch 2/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9800 - loss: 0.0994 - val_accuracy: 1.0000 - val_loss: 0.0071
Epoch 3/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9719 - loss: 0.1973 - val_accuracy: 1.0000 - val_loss: 0.0025
Epoch 4/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9858 - loss: 0.0642 - val_accuracy: 1.0000 - val_loss: 0.0015
Epoch 5/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9923 - loss: 0.0324 - val_accuracy: 1.0000 - val_loss: 0.0011
Epoch 6/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9941 - loss: 0.0252 - val_accuracy: 1.0000 - val_loss: 8.4307e-04
Epoch 7/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9980 - loss: 0.0089 - val_accuracy: 0.9976 - val_loss: 0.0057
Epoch 8/100
297/297 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step - accuracy: 0.9755 - loss: 0.189

In [16]:
BLSMOTE_metrics

{'f1_macro': 0.9766960117104666,
 'f1_weighted': 0.9843984329142723,
 'f1_micro': 0.9844097995545658,
 'roc_auc_micro': 0.9960115111862872,
 'roc_auc_per_class': {np.str_('Macrophage'): 0.9941936689004809,
  np.str_('Microglia'): 0.9997843852505374,
  np.str_('Monocyte'): 0.9948116536544485,
  np.str_('other'): 0.9841376017030709,
  'micro': 0.9960115111862872},
 'loss': 0.4859751760959625,
 'accuracy': 0.9844098091125488}

# Node Light up Test is below

In [3]:
# Import expression data and transcript data from filtered h5 file
filtered_file_tf = h5py.File("/scratch/by2372/pipeline/ARCHS4_Macrophages/Source_name_4_class/filtered_ARCHS4_TFs_4_class.h5", 'r')
ensembl_id = filtered_file_tf['ensembl_id'][...]
symbol = filtered_file_tf['symbol'][...]
expression = filtered_file_tf['data/filtered_expression'][...]
filtered_file_tf.close()

# Read all names and decode bytes to strings
if isinstance(ensembl_id[0], bytes):
    ensembl_id = [s.decode('utf-8') for s in ensembl_id]

# Read all names and decode bytes to strings
if isinstance(symbol[0], bytes):
    symbol = [s.decode('utf-8') for s in symbol]

# Make the transcript ID be the symbol _ ensembl ID
transcript_ID = []
for idx, sym in enumerate(symbol):
    ID = sym + "_" + ensembl_id[idx]
    transcript_ID.append(ID)

In [4]:
expression.shape

(13146, 10775)

In [5]:
len(transcript_ID)

13146

In [6]:
class_names = np.load('/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/unbalanced_4_class_class_names.npy')
class_names

array(['Macrophage', 'Microglia', 'Monocyte', 'other'], dtype='<U10')

### Implement the Node Light up technique as a function

In [11]:
def node_light_up(model_name,tf_names,express_mat,cell_type):
    # load train keras model
    model = load_model('/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/' + model_name + '_trained_NN.keras')
    # create square matrix of zeros. Number of rows and cols is number of TFs
    inp = np.zeros((model.input_shape[1],model.input_shape[1])).T
    # Import cell class names 
    class_names = np.load('/scratch/by2372/pipeline/ARCHS4_Macrophages/Trained_NN/' + model_name + '_class_names.npy')

    cell_type_index = np.where(class_names == cell_type)[0][0]
    # Generate baseline TF expression values for the inp square matrix
    tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)
    for i in range(inp.shape[1]):
        inp[i, :] = tf_data.values
    # Generate doubled and halved expression for TFs. Only diagonals of the baseline square matrix is doubled or halved
    half = inp.copy()
    double = inp.copy()
    half[np.eye(half.shape[0]).astype(bool)] = 0.5*half[np.eye(half.shape[0]).astype(bool)]
    double[np.eye(double.shape[0]).astype(bool)] = 2*double[np.eye(double.shape[0]).astype(bool)]

    background_pred = model.predict(inp)
    background_idx = np.argmax(background_pred, axis=1)
    background_labels = class_names[background_idx]
    
    double_pred = model.predict(double)
    double_idx = np.argmax(double_pred, axis=1)
    double_labels = class_names[double_idx]
    
    half_pred = model.predict(half)
    half_idx = np.argmax(half_pred, axis=1)
    half_labels = class_names[half_idx]

    double_background = double_pred[:,cell_type_index] - background_pred[:,cell_type_index]
    half_background = half_pred[:,cell_type_index] - background_pred[:,cell_type_index]

    score = (double_background - half_background) / 2
    score = pd.DataFrame(score, index=tf_names, columns=[cell_type])
    score_sorted = score.sort_values(by=cell_type, ascending=True)
    
    # 7th column is for macrophages classification
    #double_background = (double_pred[:,cell_type_index] - background_pred[:,cell_type_index]) / background_pred[:,cell_type_index]
    #double_background = pd.DataFrame(double_background)
    #double_background.index = tf_names
    #double_background.columns = [cell_type]
    #double_background_sorted = double_background.sort_values(by = cell_type, ascending=True)
    
    #half_background = (half_pred[:,cell_type_index] - background_pred[:,cell_type_index]) / background_pred[:,cell_type_index]
    #half_background = pd.DataFrame(half_background)
    #half_background.index = tf_names
    #half_background.columns = [cell_type]
    #half_background_sorted = half_background.sort_values(by = cell_type, ascending=True)

    return score_sorted



In [13]:
unbalanced_macrophages = node_light_up("unbalanced_4_class",transcript_ID,expression,"Macrophage")

/state/partition1/job-4470344/ipykernel_724939/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [14]:
unbalanced_macrophages.head(n=20)

,Macrophage
JUNB_ENST00000302754,-0.462145
SPI1_ENST00000378538,-0.313338
KLF6_ENST00000542957,-0.269514
CEBPB_ENST00000303004,-0.259794
NME2_ENST00000512737,-0.244788
ATF4_ENST00000674920,-0.231864
STAT6_ENST00000300134,-0.189392
KMT2A_ENST00000648029,-0.189383
FOS_ENST00000303562,-0.145038
FOS_ENST00000556324,-0.142716


In [15]:
for idx, t in enumerate(unbalanced_macrophages.index):
    if "ZNF385A" in str(t):
        print(idx)

149
257
368
557
2144
2196
2437
2812
9036
10781
11898
12270
12992
13056
13097


In [16]:
unbalanced_microglia = node_light_up("unbalanced_4_class",transcript_ID,expression,"Microglia")

/state/partition1/job-4470344/ipykernel_724939/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [17]:
unbalanced_microglia.head(n=20)

,Microglia
YBX1_ENST00000436427,-5.651181e-11
JUNB_ENST00000302754,-4.772771e-11
SPI1_ENST00000227163,-3.108271e-11
XBP1_ENST00000216037,-2.464278e-11
CEBPD_ENST00000408965,-2.393846e-11
MAFB_ENST00000373313,-2.115332e-11
FOS_ENST00000303562,-1.905499e-11
KLF6_ENST00000542957,-1.883375e-11
PLSCR1_ENST00000342435,-1.876881e-11
MSC_ENST00000325509,-1.837029e-11


In [18]:
unbalanced_monocyte = node_light_up("unbalanced_4_class",transcript_ID,expression,"Monocyte")

/state/partition1/job-4470344/ipykernel_724939/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [19]:
unbalanced_monocyte.head(n=20)

,Monocyte
YBX1_ENST00000467957,-0.184648
YBX1_ENST00000436427,-0.180192
ZNF706_ENST00000517844,-0.154406
NR1H3_ENST00000441012,-0.151993
CEBPD_ENST00000408965,-0.142018
ATF3_ENST00000341491,-0.133511
ZNF706_ENST00000519744,-0.127110
MSC_ENST00000325509,-0.106240
JUN_ENST00000678696,-0.100035
TSC22D1_ENST00000261489,-0.099088


In [20]:
for idx, t in enumerate(unbalanced_monocyte.index):
    if "ZNF385A" in str(t):
        print(idx)

48
89
153
875
1247
2364
4109
10333
10708
10949
11001
12588
12777
12888
12996


In [21]:
unbalanced_other = node_light_up("unbalanced_4_class",transcript_ID,expression,"other")

/state/partition1/job-4470344/ipykernel_724939/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [22]:
unbalanced_other.head(n=20)

,other
YBX1_ENST00000436427,-5.260964e-07
CEBPD_ENST00000408965,-3.520194e-07
YBX1_ENST00000467957,-3.264876e-07
SPI1_ENST00000227163,-3.222357e-07
HMGA1_ENST00000311487,-3.011820e-07
MSC_ENST00000325509,-2.980084e-07
MAFB_ENST00000373313,-2.824736e-07
FOS_ENST00000303562,-2.173225e-07
NR1H3_ENST00000441012,-2.154600e-07
ZNF706_ENST00000519744,-1.969321e-07


In [23]:
for idx, t in enumerate(unbalanced_other.index):
    if "ZNF385A" in str(t):
        print(idx)

20
74
204
292
1182
1379
3248
11517
11666
11905
11913
12064
12638
12879
12979


In [45]:
ROS_macrophages = node_light_up("ROS_4_class",transcript_ID,expression,"Macrophage")

/state/partition1/job-4454411/ipykernel_873094/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [46]:
ROS_macrophages.head(n=20) 

,Macrophage
FOS_ENST00000303562,-0.000005
MAFB_ENST00000373313,-0.000004
JUNB_ENST00000302754,-0.000004
KLF6_ENST00000542957,-0.000004
KMT2A_ENST00000648029,-0.000003
YBX1_ENST00000332220,-0.000003
XBP1_ENST00000216037,-0.000003
STAT6_ENST00000300134,-0.000003
KLF6_ENST00000497571,-0.000003
FOS_ENST00000556324,-0.000002


In [47]:
ROS_microglia = node_light_up("ROS_4_class",transcript_ID,expression,"Microglia")

/state/partition1/job-4454411/ipykernel_873094/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [48]:
ROS_microglia.head(n=20) # doubling

,Microglia
MAFB_ENST00000373313,-2.770355e-15
FOS_ENST00000303562,-2.722074e-15
XBP1_ENST00000216037,-2.563653e-15
KLF6_ENST00000497571,-2.372188e-15
KLF6_ENST00000542957,-2.329895e-15
MSC_ENST00000325509,-2.242758e-15
STAT6_ENST00000300134,-2.232001e-15
YBX1_ENST00000332220,-1.719414e-15
PLSCR1_ENST00000342435,-1.683094e-15
KMT2A_ENST00000648029,-1.677730e-15


In [49]:
SMOTE_macrophages = node_light_up("SMOTE_4_class",transcript_ID,expression,"Macrophage")

/state/partition1/job-4454411/ipykernel_873094/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [50]:
SMOTE_macrophages.head(n=20) 

,Macrophage
KLF6_ENST00000542957,-0.024114
FOS_ENST00000303562,-0.016860
ATF4_ENST00000674920,-0.012609
KMT2A_ENST00000648029,-0.012375
STAT6_ENST00000300134,-0.009320
XBP1_ENST00000216037,-0.008497
YBX1_ENST00000321358,-0.007506
USF2_ENST00000222305,-0.006612
NFKB1_ENST00000652569,-0.005780
SPI1_ENST00000378538,-0.005556


In [51]:
SMOTE_microglia = node_light_up("SMOTE_4_class",transcript_ID,expression,"Microglia")

/state/partition1/job-4454411/ipykernel_873094/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [52]:
SMOTE_microglia.head(n=20)

,Microglia
YBX1_ENST00000436427,-2.026894e-08
MSC_ENST00000325509,-8.206555e-09
CEBPD_ENST00000408965,-6.377741e-09
JUNB_ENST00000302754,-5.279803e-09
YBX1_ENST00000467957,-4.491543e-09
HMGA1_ENST00000311487,-4.341952e-09
NFE2L2_ENST00000397062,-4.326965e-09
PLSCR1_ENST00000342435,-4.016025e-09
ZNF706_ENST00000519744,-3.534767e-09
EGR2_ENST00000242480,-3.116971e-09


In [53]:
BLSMOTE_macrophages = node_light_up("BLSMOTE_4_class",transcript_ID,expression,"Macrophage")

/state/partition1/job-4454411/ipykernel_873094/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [54]:
BLSMOTE_macrophages.head(n=20)

,Macrophage
KMT2A_ENST00000648029,-0.413059
KLF6_ENST00000542957,-0.410554
SPI1_ENST00000378538,-0.405318
YBX1_ENST00000332220,-0.335987
STAT6_ENST00000300134,-0.317484
CEBPB_ENST00000303004,-0.282957
YBX1_ENST00000321358,-0.278203
MAFB_ENST00000373313,-0.269215
NFKB1_ENST00000652569,-0.237489
ATF4_ENST00000674920,-0.221919


In [55]:
BLSMOTE_microglia = node_light_up("BLSMOTE_4_class",transcript_ID,expression,"Microglia")

/state/partition1/job-4454411/ipykernel_873094/1994595373.py:11: Pandas4Warning: Starting with pandas version 4.0 all arguments of mean will be keyword-only.
  tf_data = pd.DataFrame(express_mat).iloc[:,:].mean(1)


411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step  
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 
411/411 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step 


In [56]:
BLSMOTE_microglia.head(n=20)

,Microglia
YBX1_ENST00000436427,-1.340101e-07
MSC_ENST00000325509,-6.143340e-08
YBX1_ENST00000467957,-5.877298e-08
CEBPD_ENST00000408965,-4.634818e-08
NR1H3_ENST00000441012,-4.275080e-08
ZNF706_ENST00000517844,-3.876193e-08
IRF1_ENST00000405885,-3.841902e-08
ZNF706_ENST00000519744,-3.813847e-08
HMGA1_ENST00000311487,-3.758130e-08
ZNF385A_ENST00000394313,-2.923500e-08
